# Yelp Metadata and Knowledge Graph Preparation

This notebook prepares structured business information and the knowledge graph
used by the KGRec-inspired recommendation experiment.

Only features that can be methodologically justified as business attributes are
used as structured recommendation evidence. Snapshot popularity and outcome
variables are audited separately to avoid introducing unintended leakage or
confounding into the controlled KGRec-NV versus KGRec-MM comparison.

In [68]:
from pathlib import Path
import pandas as pd
import numpy as np

processed_data_dir = Path(
    "processed_data/new_orleans_subset"
)

personalisation_businesses = pd.read_parquet(
    processed_data_dir
    / "new_orleans_personalisation_businesses.parquet"
)

print(
    "Business rows:",
    f"{len(personalisation_businesses):,}"
)

print(
    "Unique business IDs:",
    f"{personalisation_businesses['business_id'].nunique():,}"
)

print("\nColumns:")
for column in personalisation_businesses.columns:
    print(column)

Business rows: 2,516
Unique business IDs: 2,516

Columns:
business_id
name
address
city
state
postal_code
latitude
longitude
stars
review_count
is_open
attributes
categories
hours
city_clean
state_clean
category_list
matched_target_categories
is_target_business
is_operational_at_snapshot


In [69]:
metadata_schema_audit = pd.DataFrame({
    "column":
        personalisation_businesses.columns,

    "dtype": [
        str(
            personalisation_businesses[
                column
            ].dtype
        )
        for column
        in personalisation_businesses.columns
    ],

    "missing_count": [
        personalisation_businesses[
            column
        ].isna().sum()
        for column
        in personalisation_businesses.columns
    ],

    "missing_percentage": [
        round(
            personalisation_businesses[
                column
            ].isna().mean()
            * 100,
            2
        )
        for column
        in personalisation_businesses.columns
    ],

    "unique_values": [
        personalisation_businesses[
            column
        ].nunique(
            dropna=True
        )
        for column
        in personalisation_businesses.columns
    ]
})

metadata_schema_audit

,column,dtype,missing_count,missing_percentage,unique_values
0,business_id,object,0,0.00,2516
1,name,object,0,0.00,2321
2,address,object,0,0.00,1977
3,city,object,0,0.00,2
4,state,object,0,0.00,1
5,postal_code,object,0,0.00,31
6,latitude,float64,0,0.00,2339
7,longitude,float64,0,0.00,2284
8,stars,float64,0,0.00,8
9,review_count,int64,0,0.00,617


In [70]:
for column in personalisation_businesses.columns:

    print(
        "\n",
        "=" * 70
    )

    print(
        "COLUMN:",
        column
    )

    print(
        personalisation_businesses[
            column
        ]
        .dropna()
        .head(5)
        .tolist()
    )


COLUMN: business_id
['-0__F9fnKt8uioCKztF5Ww', '-1XSzguS6XLN-V6MVZMg2A', '-4x3pVUUsfWmKEilWKsOZQ', '-A2OLubXDsMRPNN7LqohPA', '-AaxZJ_I4rSFOBJbBz4SlQ']

COLUMN: name
['Piscobar', 'Restaurant Rebirth', "Dickey's Barbecue Pit", 'Rollin Fatties', "Hansen's Sno-Bliz"]

COLUMN: address
['914 Union St', '857 Fulton St', '6005 Jefferson Hwy', '1430 Tulane Ave', '4801 Tchoupitoulas St']

COLUMN: city
['New Orleans', 'New Orleans', 'New Orleans', 'New Orleans', 'New Orleans']

COLUMN: state
['LA', 'LA', 'LA', 'LA', 'LA']

COLUMN: postal_code
['70112', '70130', '70123', '70112', '70115']

COLUMN: latitude
[29.9516963, 29.9435371, 29.9415170538, 29.95504, 29.9170838415]

COLUMN: longitude
[-90.073235, -90.0653721, -90.1889420124, -90.0768769, -90.1054047793]

COLUMN: stars
[4.0, 4.5, 2.5, 5.0, 4.5]

COLUMN: review_count
[66, 521, 75, 171, 512]

COLUMN: is_open
[1, 1, 1, 1, 1]

COLUMN: attributes
['{"RestaurantsAttire": "\'casual\'", "WheelchairAccessible": "True", "BikeParking": "True", "Restaura

In [71]:
key_metadata_columns = [
    "attributes",
    "categories",
    "category_list",
    "matched_target_categories",
    "hours"
]

for column in key_metadata_columns:

    non_missing = (
        personalisation_businesses[
            column
        ]
        .dropna()
    )

    print(
        f"\n{column}"
    )

    print(
        "Python types:"
    )

    print(
        non_missing
        .map(type)
        .value_counts()
    )

    print(
        "\nExample:"
    )

    print(
        non_missing.iloc[0]
        if len(non_missing) > 0
        else None
    )


attributes
Python types:
attributes
<class 'str'>    2502
Name: count, dtype: int64

Example:
{"RestaurantsAttire": "'casual'", "WheelchairAccessible": "True", "BikeParking": "True", "RestaurantsReservations": "False", "RestaurantsDelivery": "False", "Smoking": "u'outdoor'", "RestaurantsPriceRange2": "2", "BusinessAcceptsCreditCards": "True", "RestaurantsTableService": "False", "RestaurantsGoodForGroups": "True", "GoodForDancing": "False", "HappyHour": "True", "CoatCheck": "False", "HasTV": "False", "RestaurantsTakeOut": "False", "WiFi": "u'free'", "Caters": "False", "GoodForKids": "False", "Music": "{'dj': False, 'background_music': False, 'no_music': False, 'jukebox': False, 'live': False, 'video': False, 'karaoke': False}", "DogsAllowed": "True", "GoodForMeal": "{'dessert': False, 'latenight': False, 'lunch': False, 'dinner': False, 'brunch': False, 'breakfast': False}", "OutdoorSeating": "True", "Alcohol": "u'full_bar'", "BusinessParking": "{'garage': None, 'street': True, 'valida

In [72]:
all_categories = (
    personalisation_businesses[
        "category_list"
    ]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)

category_frequency = (
    all_categories
    .value_counts()
    .rename_axis(
        "category"
    )
    .reset_index(
        name="business_count"
    )
)

category_frequency[
    "business_percentage"
] = (
    category_frequency[
        "business_count"
    ]
    /
    len(
        personalisation_businesses
    )
    * 100
).round(2)


category_audit = pd.Series({
    "Businesses":
        len(
            personalisation_businesses
        ),

    "Unique categories":
        all_categories.nunique(),

    "Total business-category links":
        len(
            all_categories
        ),

    "Mean categories per business":
        len(
            all_categories
        )
        /
        len(
            personalisation_businesses
        ),

    "Minimum categories per business":
        personalisation_businesses[
            "category_list"
        ]
        .map(len)
        .min(),

    "Maximum categories per business":
        personalisation_businesses[
            "category_list"
        ]
        .map(len)
        .max()
})

category_audit

Businesses                         2516.0
Unique categories                  2072.0
Total business-category links      2516.0
Mean categories per business          1.0
Minimum categories per business       8.0
Maximum categories per business     344.0
dtype: float64

In [73]:
category_frequency.head(30)

,category,business_count,business_percentage
0,"[""Coffee & Tea"", ""Food""]",24,0.95
1,"[""Bars"", ""Nightlife""]",22,0.87
2,"[""Nightlife"", ""Bars""]",19,0.76
3,"[""Hotels"", ""Event Planning & Services"", ""Hotel...",19,0.76
4,"[""Food"", ""Coffee & Tea""]",18,0.72
5,"[""Event Planning & Services"", ""Hotels"", ""Hotel...",14,0.56
6,"[""American (New)"", ""Restaurants""]",13,0.52
7,"[""Event Planning & Services"", ""Hotels & Travel...",12,0.48
8,"[""Restaurants"", ""Cajun/Creole""]",11,0.44
9,"[""Food"", ""Ice Cream & Frozen Yogurt""]",11,0.44


In [74]:
print(
    personalisation_businesses[
        "attributes"
    ]
    .dropna()
    .map(type)
    .value_counts()
)

print(
    "\nFirst non-missing attributes value:"
)

example_attributes = (
    personalisation_businesses[
        "attributes"
    ]
    .dropna()
    .iloc[0]
)

print(example_attributes)
print("\nPython type:", type(example_attributes))

attributes
<class 'str'>    2502
Name: count, dtype: int64

First non-missing attributes value:
{"RestaurantsAttire": "'casual'", "WheelchairAccessible": "True", "BikeParking": "True", "RestaurantsReservations": "False", "RestaurantsDelivery": "False", "Smoking": "u'outdoor'", "RestaurantsPriceRange2": "2", "BusinessAcceptsCreditCards": "True", "RestaurantsTableService": "False", "RestaurantsGoodForGroups": "True", "GoodForDancing": "False", "HappyHour": "True", "CoatCheck": "False", "HasTV": "False", "RestaurantsTakeOut": "False", "WiFi": "u'free'", "Caters": "False", "GoodForKids": "False", "Music": "{'dj': False, 'background_music': False, 'no_music': False, 'jukebox': False, 'live': False, 'video': False, 'karaoke': False}", "DogsAllowed": "True", "GoodForMeal": "{'dessert': False, 'latenight': False, 'lunch': False, 'dinner': False, 'brunch': False, 'breakfast': False}", "OutdoorSeating": "True", "Alcohol": "u'full_bar'", "BusinessParking": "{'garage': None, 'street': True, 'valid

In [75]:
import ast


def parse_yelp_dict(value):
    """
    Safely convert Yelp dictionary-like values to Python dicts.

    Returns an empty dict when the value is missing,
    malformed, or not dictionary-like.
    """

    if isinstance(value, dict):
        return value

    if pd.isna(value):
        return {}

    if isinstance(value, str):

        value = value.strip()

        if value == "":
            return {}

        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, dict):
                return parsed

        except (
            ValueError,
            SyntaxError
        ):
            pass

    return {}

In [76]:
personalisation_businesses[
    "attributes_parsed"
] = (
    personalisation_businesses[
        "attributes"
    ]
    .apply(
        parse_yelp_dict
    )
)

In [77]:
attribute_parsing_check = pd.Series({
    "Businesses":
        len(
            personalisation_businesses
        ),

    "Original missing attributes":
        personalisation_businesses[
            "attributes"
        ].isna().sum(),

    "Parsed dictionaries":
        personalisation_businesses[
            "attributes_parsed"
        ]
        .map(
            lambda x:
                isinstance(x, dict)
        )
        .sum(),

    "Non-empty parsed dictionaries":
        personalisation_businesses[
            "attributes_parsed"
        ]
        .map(
            lambda x:
                len(x) > 0
        )
        .sum(),

    "Empty parsed dictionaries":
        personalisation_businesses[
            "attributes_parsed"
        ]
        .map(
            lambda x:
                len(x) == 0
        )
        .sum()
})

attribute_parsing_check

Businesses                       2516
Original missing attributes        14
Parsed dictionaries              2516
Non-empty parsed dictionaries    2502
Empty parsed dictionaries          14
dtype: int64

In [78]:
attribute_key_records = []

for row in (
    personalisation_businesses[
        [
            "business_id",
            "attributes_parsed"
        ]
    ]
    .itertuples(
        index=False
    )
):

    attributes = (
        row.attributes_parsed
    )

    for key in attributes.keys():

        attribute_key_records.append({
            "business_id":
                row.business_id,

            "attribute":
                key
        })


attribute_key_df = pd.DataFrame(
    attribute_key_records,
    columns=[
        "business_id",
        "attribute"
    ]
)


attribute_key_frequency = (
    attribute_key_df[
        "attribute"
    ]
    .value_counts()
    .rename_axis(
        "attribute"
    )
    .reset_index(
        name="business_count"
    )
)


attribute_key_frequency[
    "business_percentage"
] = (
    attribute_key_frequency[
        "business_count"
    ]
    /
    len(
        personalisation_businesses
    )
    * 100
).round(2)


print(
    "Attribute records:",
    f"{len(attribute_key_df):,}"
)

print(
    "Unique top-level attribute keys:",
    attribute_key_df[
        "attribute"
    ].nunique()
)

attribute_key_frequency

Attribute records: 40,117
Unique top-level attribute keys: 38


,attribute,business_count,business_percentage
0,BusinessAcceptsCreditCards,2393,95.11
1,RestaurantsPriceRange2,2247,89.31
2,BusinessParking,2234,88.79
3,OutdoorSeating,1963,78.02
4,WiFi,1934,76.87
5,RestaurantsTakeOut,1922,76.39
6,BikeParking,1902,75.60
7,Ambience,1816,72.18
8,HasTV,1816,72.18
9,Alcohol,1811,71.98


In [79]:
nested_attribute_examples = []

for attributes in (
    personalisation_businesses[
        "attributes"
    ]
    .dropna()
):

    if not isinstance(
        attributes,
        dict
    ):
        continue

    for key, value in attributes.items():

        if isinstance(
            value,
            dict
        ):

            nested_attribute_examples.append({
                "attribute":
                    key,

                "nested_keys":
                    list(
                        value.keys()
                    )
            })


nested_attribute_df = pd.DataFrame(
    nested_attribute_examples
)


if len(
    nested_attribute_df
) > 0:

    nested_attribute_summary = (
        nested_attribute_df[
            "attribute"
        ]
        .value_counts()
    )

    print(
        "Nested attributes:"
    )

    print(
        nested_attribute_summary
    )

    print(
        "\nExamples:"
    )

    print(
        nested_attribute_df
        .drop_duplicates(
            subset="attribute"
        )
        .head(20)
    )

else:

    print(
        "No dictionary-valued nested "
        "attributes detected."
    )

No dictionary-valued nested attributes detected.


In [80]:
attribute_value_records = []

for row in (
    personalisation_businesses[
        [
            "business_id",
            "attributes_parsed"
        ]
    ]
    .itertuples(
        index=False
    )
):

    for key, value in row.attributes_parsed.items():

        attribute_value_records.append({
            "business_id":
                row.business_id,

            "attribute":
                key,

            "raw_value":
                value,

            "raw_type":
                type(value).__name__
        })


attribute_value_df = pd.DataFrame(
    attribute_value_records
)


attribute_value_type_summary = (
    attribute_value_df
    .groupby(
        [
            "attribute",
            "raw_type"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        [
            "attribute",
            "count"
        ],
        ascending=[
            True,
            False
        ]
    )
)


attribute_value_type_summary

,attribute,raw_type,count
0,AcceptsInsurance,str,1
1,AgesAllowed,str,6
2,Alcohol,str,1811
3,Ambience,str,1816
4,BYOB,str,260
5,BYOBCorkage,str,70
6,BestNights,str,473
7,BikeParking,str,1902
8,BusinessAcceptsBitcoin,str,539
9,BusinessAcceptsCreditCards,str,2393


In [81]:
def parse_nested_attribute(value):

    # Already genuinely nested
    if isinstance(value, dict):
        return value

    if not isinstance(value, str):
        return None

    value_clean = value.strip()

    if not (
        value_clean.startswith("{")
        and value_clean.endswith("}")
    ):
        return None

    try:
        parsed = ast.literal_eval(
            value_clean
        )

        if isinstance(parsed, dict):
            return parsed

    except (
        ValueError,
        SyntaxError
    ):
        pass

    return None

In [82]:
nested_attribute_records = []

for row in (
    attribute_value_df
    .itertuples(
        index=False
    )
):

    nested_value = parse_nested_attribute(
        row.raw_value
    )

    if nested_value is not None:

        nested_attribute_records.append({
            "business_id":
                row.business_id,

            "attribute":
                row.attribute,

            "nested_key_count":
                len(
                    nested_value
                ),

            "nested_keys":
                list(
                    nested_value.keys()
                )
        })


nested_attribute_df = pd.DataFrame(
    nested_attribute_records
)


nested_attribute_summary = (
    nested_attribute_df[
        "attribute"
    ]
    .value_counts()
    .rename_axis(
        "attribute"
    )
    .reset_index(
        name="business_count"
    )
)


nested_attribute_summary

,attribute,business_count
0,BusinessParking,2216
1,Ambience,1811
2,GoodForMeal,1190
3,Music,564
4,BestNights,472
5,DietaryRestrictions,1


In [83]:
for attribute_name in (
    nested_attribute_summary[
        "attribute"
    ]
):

    example = (
        attribute_value_df.loc[
            attribute_value_df[
                "attribute"
            ]
            == attribute_name,
            "raw_value"
        ]
        .dropna()
        .iloc[0]
    )

    print(
        "\n",
        "=" * 70
    )

    print(
        attribute_name
    )

    print(
        parse_nested_attribute(
            example
        )
    )


BusinessParking
{'garage': None, 'street': True, 'validated': None, 'lot': None, 'valet': False}

Ambience
{'touristy': False, 'hipster': True, 'romantic': None, 'divey': False, 'intimate': None, 'trendy': None, 'upscale': None, 'classy': None, 'casual': True}

GoodForMeal
{'dessert': False, 'latenight': False, 'lunch': False, 'dinner': False, 'brunch': False, 'breakfast': False}

Music
{'dj': False, 'background_music': False, 'no_music': False, 'jukebox': False, 'live': False, 'video': False, 'karaoke': False}

BestNights
{'monday': False, 'tuesday': False, 'friday': False, 'wednesday': False, 'thursday': False, 'sunday': False, 'saturday': True}

DietaryRestrictions
{'dairy-free': False, 'gluten-free': False, 'vegan': True, 'kosher': False, 'halal': True, 'soy-free': False, 'vegetarian': True}


In [84]:
scalar_attribute_df = (
    attribute_value_df[
        attribute_value_df[
            "raw_value"
        ]
        .map(
            parse_nested_attribute
        )
        .isna()
    ]
    .copy()
)

In [85]:
scalar_value_frequency = (
    scalar_attribute_df
    .groupby(
        [
            "attribute",
            "raw_value"
        ]
    )
    .size()
    .reset_index(
        name="business_count"
    )
    .sort_values(
        [
            "attribute",
            "business_count"
        ],
        ascending=[
            True,
            False
        ]
    )
)


for attribute_name in (
    scalar_value_frequency[
        "attribute"
    ]
    .drop_duplicates()
):

    print(
        "\n",
        "=" * 70
    )

    print(
        attribute_name
    )

    print(
        scalar_value_frequency[
            scalar_value_frequency[
                "attribute"
            ]
            == attribute_name
        ]
        .head(10)
        .to_string(
            index=False
        )
    )


AcceptsInsurance
       attribute raw_value  business_count
AcceptsInsurance     False               1

AgesAllowed
  attribute  raw_value  business_count
AgesAllowed  u'21plus'               3
AgesAllowed u'allages'               2
AgesAllowed  u'18plus'               1

Alcohol
attribute        raw_value  business_count
  Alcohol      u'full_bar'             990
  Alcohol          u'none'             340
  Alcohol       'full_bar'             220
  Alcohol u'beer_and_wine'             110
  Alcohol           'none'             109
  Alcohol  'beer_and_wine'              42

Ambience
attribute raw_value  business_count
 Ambience      None               5

BYOB
attribute raw_value  business_count
     BYOB     False             213
     BYOB      True              47

BYOBCorkage
  attribute      raw_value  business_count
BYOBCorkage           'no'              36
BYOBCorkage     'yes_free'              19
BYOBCorkage  'yes_corkage'              12
BYOBCorkage          u'no'          

In [86]:
all_categories = (
    personalisation_businesses[
        "category_list"
    ]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)


category_frequency = (
    all_categories
    .value_counts()
    .rename_axis(
        "category"
    )
    .reset_index(
        name="business_count"
    )
)


category_frequency[
    "business_percentage"
] = (
    category_frequency[
        "business_count"
    ]
    /
    len(
        personalisation_businesses
    )
    * 100
).round(2)


categories_per_business = (
    personalisation_businesses[
        "category_list"
    ]
    .map(len)
)


category_audit = pd.Series({
    "Businesses":
        len(
            personalisation_businesses
        ),

    "Unique categories":
        all_categories.nunique(),

    "Total business-category links":
        len(
            all_categories
        ),

    "Mean categories per business":
        categories_per_business.mean(),

    "Median categories per business":
        categories_per_business.median(),

    "Minimum categories per business":
        categories_per_business.min(),

    "Maximum categories per business":
        categories_per_business.max(),

    "Categories appearing once":
        (
            category_frequency[
                "business_count"
            ]
            == 1
        ).sum(),

    "Categories appearing in <5 businesses":
        (
            category_frequency[
                "business_count"
            ]
            < 5
        ).sum()
})

category_audit

Businesses                               2516.000000
Unique categories                        2072.000000
Total business-category links            2516.000000
Mean categories per business               73.202305
Median categories per business             65.000000
Minimum categories per business             8.000000
Maximum categories per business           344.000000
Categories appearing once                1948.000000
Categories appearing in <5 businesses    2039.000000
dtype: float64

In [87]:
category_frequency.head(5)

,category,business_count,business_percentage
0,"[""Coffee & Tea"", ""Food""]",24,0.95
1,"[""Bars"", ""Nightlife""]",22,0.87
2,"[""Nightlife"", ""Bars""]",19,0.76
3,"[""Hotels"", ""Event Planning & Services"", ""Hotel...",19,0.76
4,"[""Food"", ""Coffee & Tea""]",18,0.72


In [88]:
print(
    personalisation_businesses[
        "category_list"
    ]
    .map(type)
    .value_counts()
)

print(
    "\nExample category_list value:"
)

example_categories = (
    personalisation_businesses[
        "category_list"
    ]
    .iloc[0]
)

print(example_categories)
print(
    "Python type:",
    type(example_categories)
)

category_list
<class 'str'>    2516
Name: count, dtype: int64

Example category_list value:
["Cafes", "Nightlife", "Cocktail Bars", "Peruvian", "Restaurants", "Vegan", "Bars"]
Python type: <class 'str'>


In [89]:
import ast


def parse_yelp_list(value):
    """
    Safely parse Yelp list-like values.

    Returns an empty list for missing,
    malformed or non-list values.
    """

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if isinstance(value, str):

        value = value.strip()

        if value == "":
            return []

        try:
            parsed = ast.literal_eval(
                value
            )

            if isinstance(parsed, list):
                return parsed

        except (
            ValueError,
            SyntaxError
        ):
            pass

    return []

In [90]:
personalisation_businesses[
    "category_list_parsed"
] = (
    personalisation_businesses[
        "category_list"
    ]
    .apply(
        parse_yelp_list
    )
)

In [91]:
category_parsing_check = pd.Series({
    "Businesses":
        len(
            personalisation_businesses
        ),

    "Parsed lists":
        personalisation_businesses[
            "category_list_parsed"
        ]
        .map(
            lambda x:
                isinstance(x, list)
        )
        .sum(),

    "Non-empty category lists":
        personalisation_businesses[
            "category_list_parsed"
        ]
        .map(len)
        .gt(0)
        .sum(),

    "Empty category lists":
        personalisation_businesses[
            "category_list_parsed"
        ]
        .map(len)
        .eq(0)
        .sum(),

    "Minimum categories per business":
        personalisation_businesses[
            "category_list_parsed"
        ]
        .map(len)
        .min(),

    "Maximum categories per business":
        personalisation_businesses[
            "category_list_parsed"
        ]
        .map(len)
        .max()
})

category_parsing_check

Businesses                         2516
Parsed lists                       2516
Non-empty category lists           2516
Empty category lists                  0
Minimum categories per business       1
Maximum categories per business      19
dtype: int64

In [92]:
all_categories = (
    personalisation_businesses[
        "category_list_parsed"
    ]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)

all_categories = (
    all_categories[
        all_categories != ""
    ]
)

In [93]:
category_frequency = (
    all_categories
    .value_counts()
    .rename_axis(
        "category"
    )
    .reset_index(
        name="business_count"
    )
)


category_frequency[
    "business_percentage"
] = (
    category_frequency[
        "business_count"
    ]
    /
    len(
        personalisation_businesses
    )
    * 100
).round(2)

In [94]:
categories_per_business = (
    personalisation_businesses[
        "category_list_parsed"
    ]
    .map(len)
)


category_audit = pd.Series({
    "Businesses":
        len(
            personalisation_businesses
        ),

    "Unique categories":
        all_categories.nunique(),

    "Total business-category links":
        len(
            all_categories
        ),

    "Mean categories per business":
        categories_per_business.mean(),

    "Median categories per business":
        categories_per_business.median(),

    "Minimum categories per business":
        categories_per_business.min(),

    "Maximum categories per business":
        categories_per_business.max(),

    "Categories appearing once":
        (
            category_frequency[
                "business_count"
            ]
            == 1
        ).sum(),

    "Categories appearing in <5 businesses":
        (
            category_frequency[
                "business_count"
            ]
            < 5
        ).sum()
})

category_audit

Businesses                                2516.000000
Unique categories                          375.000000
Total business-category links            12554.000000
Mean categories per business                 4.989666
Median categories per business               5.000000
Minimum categories per business              1.000000
Maximum categories per business             19.000000
Categories appearing once                  100.000000
Categories appearing in <5 businesses      192.000000
dtype: float64

In [95]:
category_frequency.head(10)

,category,business_count,business_percentage
0,Restaurants,1734,68.92
1,Food,916,36.41
2,Nightlife,754,29.97
3,Bars,684,27.19
4,Event Planning & Services,353,14.03
5,Cajun/Creole,340,13.51
6,Breakfast & Brunch,314,12.48
7,Seafood,286,11.37
8,American (New),271,10.77
9,Sandwiches,265,10.53


## Stage F1.5 — Category Frequency Threshold Audit

The Yelp category vocabulary contains a long tail of very rare categories.
Because knowledge-graph entities that occur for only a small number of
businesses provide limited shared connectivity, alternative minimum-frequency
thresholds are compared before finalising the graph schema.

The threshold is selected based on the trade-off between reducing sparse
category entities and retaining business-category evidence.

In [96]:
category_threshold_results = []

for min_businesses in [
    1,
    2,
    5,
    10
]:

    retained_categories = set(
        category_frequency.loc[
            category_frequency[
                "business_count"
            ] >= min_businesses,
            "category"
        ]
    )

    retained_links = (
        personalisation_businesses[
            [
                "business_id",
                "category_list_parsed"
            ]
        ]
        .explode(
            "category_list_parsed"
        )
        .rename(
            columns={
                "category_list_parsed":
                    "category"
            }
        )
    )

    retained_links[
        "category"
    ] = (
        retained_links[
            "category"
        ]
        .astype(str)
        .str.strip()
    )

    retained_links = (
        retained_links[
            retained_links[
                "category"
            ].isin(
                retained_categories
            )
        ]
    )

    businesses_with_category = (
        retained_links[
            "business_id"
        ]
        .nunique()
    )

    category_threshold_results.append({
        "minimum_business_frequency":
            min_businesses,

        "retained_categories":
            len(
                retained_categories
            ),

        "retained_category_percentage":
            round(
                len(
                    retained_categories
                )
                /
                category_frequency[
                    "category"
                ].nunique()
                * 100,
                2
            ),

        "retained_links":
            len(
                retained_links
            ),

        "retained_link_percentage":
            round(
                len(
                    retained_links
                )
                /
                len(
                    all_categories
                )
                * 100,
                2
            ),

        "businesses_with_at_least_one_category":
            businesses_with_category,

        "business_coverage_percentage":
            round(
                businesses_with_category
                /
                len(
                    personalisation_businesses
                )
                * 100,
                2
            )
    })


category_threshold_comparison = (
    pd.DataFrame(
        category_threshold_results
    )
)

category_threshold_comparison

,minimum_business_frequency,retained_categories,retained_category_percentage,retained_links,retained_link_percentage,businesses_with_at_least_one_category,business_coverage_percentage
0,1,375,100.00,12554,100.00,2516,100.0
1,2,275,73.33,12454,99.20,2516,100.0
2,5,183,48.80,12203,97.20,2516,100.0
3,10,120,32.00,11775,93.79,2516,100.0


In [97]:
MIN_CATEGORY_FREQUENCY = 5

candidate_categories = set(
    category_frequency.loc[
        category_frequency[
            "business_count"
        ] >= MIN_CATEGORY_FREQUENCY,
        "category"
    ]
)


business_category_counts_after_filter = (
    personalisation_businesses[
        [
            "business_id",
            "category_list_parsed"
        ]
    ]
    .explode(
        "category_list_parsed"
    )
    .rename(
        columns={
            "category_list_parsed":
                "category"
        }
    )
)


business_category_counts_after_filter[
    "category"
] = (
    business_category_counts_after_filter[
        "category"
    ]
    .astype(str)
    .str.strip()
)


business_category_counts_after_filter = (
    business_category_counts_after_filter[
        business_category_counts_after_filter[
            "category"
        ].isin(
            candidate_categories
        )
    ]
    .groupby(
        "business_id"
    )
    .size()
)


businesses_without_retained_category = (
    set(
        personalisation_businesses[
            "business_id"
        ]
    )
    -
    set(
        business_category_counts_after_filter.index
    )
)


print(
    "Businesses without any category after ≥5 filter:",
    len(
        businesses_without_retained_category
    )
)

Businesses without any category after ≥5 filter: 0


## Stage F1.6 — Freeze the category component

First save the final retained category vocabulary and business-category links.

In [98]:
MIN_CATEGORY_FREQUENCY = 5

retained_category_frequency = (
    category_frequency[
        category_frequency[
            "business_count"
        ] >= MIN_CATEGORY_FREQUENCY
    ]
    .copy()
    .reset_index(drop=True)
)

retained_categories = set(
    retained_category_frequency[
        "category"
    ]
)

print(
    "Retained categories:",
    len(retained_categories)
)

Retained categories: 183


In [99]:
business_category_links = (
    personalisation_businesses[
        [
            "business_id",
            "category_list_parsed"
        ]
    ]
    .explode(
        "category_list_parsed"
    )
    .rename(
        columns={
            "category_list_parsed":
                "category"
        }
    )
    .copy()
)

business_category_links[
    "category"
] = (
    business_category_links[
        "category"
    ]
    .astype(str)
    .str.strip()
)

business_category_links = (
    business_category_links[
        business_category_links[
            "category"
        ].isin(
            retained_categories
        )
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Final business-category links:",
    f"{len(business_category_links):,}"
)

print(
    "Businesses represented:",
    business_category_links[
        "business_id"
    ].nunique()
)

print(
    "Categories represented:",
    business_category_links[
        "category"
    ].nunique()
)

Final business-category links: 12,203
Businesses represented: 2516
Categories represented: 183


In [100]:
category_component_check = pd.Series({
    "Personalisation businesses":
        personalisation_businesses[
            "business_id"
        ].nunique(),

    "Businesses with retained category":
        business_category_links[
            "business_id"
        ].nunique(),

    "Retained categories":
        business_category_links[
            "category"
        ].nunique(),

    "Business-category links":
        len(
            business_category_links
        ),

    "Duplicate links":
        business_category_links
        .duplicated(
            subset=[
                "business_id",
                "category"
            ]
        )
        .sum(),

    "All businesses represented":
        (
            business_category_links[
                "business_id"
            ].nunique()
            ==
            personalisation_businesses[
                "business_id"
            ].nunique()
        )
})

category_component_check

Personalisation businesses            2516
Businesses with retained category     2516
Retained categories                    183
Business-category links              12203
Duplicate links                          0
All businesses represented            True
dtype: object

## Stage F1.7 — Normalise attribute values

Before creating the actual graph triples, let's convert the Yelp string values into clean Python values.

In [101]:
def normalise_yelp_scalar(value):
    """
    Normalise common scalar Yelp attribute values.
    """

    if value is None:
        return None

    # Already a proper Boolean/number
    if isinstance(
        value,
        (
            bool,
            int,
            float
        )
    ):
        return value

    if not isinstance(
        value,
        str
    ):
        return value

    cleaned = value.strip()

    if cleaned in {
        "",
        "None",
        "none",
        "null",
        "NULL"
    }:
        return None

    # Boolean strings
    if cleaned == "True":
        return True

    if cleaned == "False":
        return False

    # Yelp sometimes stores strings like
    # u'free', 'casual', etc.
    try:
        parsed = ast.literal_eval(
            cleaned
        )

        # Avoid returning dict here;
        # nested dictionaries are handled separately.
        if not isinstance(
            parsed,
            dict
        ):
            return parsed

    except (
        ValueError,
        SyntaxError
    ):
        pass

    return cleaned

In [102]:
scalar_attribute_records = []

for row in (
    personalisation_businesses[
        [
            "business_id",
            "attributes_parsed"
        ]
    ]
    .itertuples(
        index=False
    )
):

    for attribute, raw_value in (
        row.attributes_parsed.items()
    ):

        nested_value = (
            parse_nested_attribute(
                raw_value
            )
        )

        if nested_value is not None:
            continue

        normalised_value = (
            normalise_yelp_scalar(
                raw_value
            )
        )

        scalar_attribute_records.append({
            "business_id":
                row.business_id,

            "attribute":
                attribute,

            "raw_value":
                raw_value,

            "normalised_value":
                normalised_value,

            "normalised_type":
                type(
                    normalised_value
                ).__name__
                if normalised_value
                is not None
                else "None"
        })


scalar_attribute_df = pd.DataFrame(
    scalar_attribute_records
)

In [103]:
scalar_attribute_type_summary = (
    scalar_attribute_df
    .groupby(
        [
            "attribute",
            "normalised_type"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        [
            "attribute",
            "count"
        ],
        ascending=[
            True,
            False
        ]
    )
)

scalar_attribute_type_summary

,attribute,normalised_type,count
0,AcceptsInsurance,bool,1
1,AgesAllowed,str,6
2,Alcohol,str,1811
3,Ambience,None,5
4,BYOB,bool,260
5,BYOBCorkage,str,70
6,BestNights,None,1
8,BikeParking,bool,1901
7,BikeParking,None,1
9,BusinessAcceptsBitcoin,bool,539


In [104]:
scalar_attribute_cardinality = (
    scalar_attribute_df
    .groupby(
        "attribute"
    )
    .agg(
        business_count=(
            "business_id",
            "nunique"
        ),

        unique_values=(
            "normalised_value",
            lambda x:
                x.dropna().nunique()
        ),

        missing_or_none=(
            "normalised_value",
            lambda x:
                x.isna().sum()
        )
    )
    .reset_index()
    .sort_values(
        "business_count",
        ascending=False
    )
)

scalar_attribute_cardinality

,attribute,business_count,unique_values,missing_or_none
9,BusinessAcceptsCreditCards,2393,2,0
30,RestaurantsPriceRange2,2247,4,0
25,OutdoorSeating,1963,2,94
36,WiFi,1934,3,0
33,RestaurantsTakeOut,1922,2,72
7,BikeParking,1902,2,1
21,HasTV,1816,2,0
2,Alcohol,1811,3,0
28,RestaurantsDelivery,1760,2,135
31,RestaurantsReservations,1748,2,10


In [105]:
attributes_to_inspect = [
    "RestaurantsPriceRange2",
    "Alcohol",
    "WiFi",
    "NoiseLevel",
    "RestaurantsAttire",
    "Smoking",
    "BYOBCorkage",
    "AgesAllowed"
]

for attribute_name in attributes_to_inspect:

    values = (
        scalar_attribute_df.loc[
            scalar_attribute_df[
                "attribute"
            ]
            == attribute_name,
            "normalised_value"
        ]
        .value_counts(
            dropna=False
        )
    )

    print(
        "\n",
        "=" * 60
    )

    print(attribute_name)

    print(
        values.head(20)
    )


RestaurantsPriceRange2
normalised_value
2    1374
1     692
3     171
4      10
Name: count, dtype: int64

Alcohol
normalised_value
full_bar         1210
none              449
beer_and_wine     152
Name: count, dtype: int64

WiFi
normalised_value
free    1108
no       790
paid      36
Name: count, dtype: int64

NoiseLevel
normalised_value
average      1215
quiet         235
loud          175
very_loud      61
Name: count, dtype: int64

RestaurantsAttire
normalised_value
casual    1333
dressy      80
formal       3
Name: count, dtype: int64

Smoking
normalised_value
outdoor    159
no         130
yes         26
Name: count, dtype: int64

BYOBCorkage
normalised_value
no             37
yes_free       20
yes_corkage    13
Name: count, dtype: int64

AgesAllowed
normalised_value
21plus     3
allages    2
18plus     1
Name: count, dtype: int64


In [106]:
boolean_attribute_summary = (
    scalar_attribute_df[
        scalar_attribute_df[
            "normalised_type"
        ] == "bool"
    ]
    .groupby(
        [
            "attribute",
            "normalised_value"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .rename(
        columns={
            False: "false_count",
            True: "true_count"
        }
    )
    .reset_index()
)

boolean_attribute_summary

normalised_value,attribute,false_count,true_count
0,AcceptsInsurance,1,0
1,BYOB,213,47
2,BikeParking,323,1578
3,BusinessAcceptsBitcoin,533,6
4,BusinessAcceptsCreditCards,106,2287
5,ByAppointmentOnly,315,27
6,Caters,718,862
7,CoatCheck,338,12
8,Corkage,110,110
9,DogsAllowed,623,285


## Stage F2.1 — Candidate Structured Knowledge-Graph Construction

Structured business knowledge is converted into typed knowledge-graph triples.

The graph represents:

- retained Yelp business categories;
- scalar Boolean and categorical business attributes; and
- flattened nested attributes such as ambience, parking and meal suitability.

Explicit positive and negative attribute values are preserved, while missing or
unknown values create no graph edge.

Category entities are restricted to categories occurring for at least five
businesses, based on the preceding threshold analysis.

This stage constructs and audits candidate metadata triples before the final
knowledge graph is frozen and combined with training-only user–business
interaction edges.

In [107]:
def canonical_kg_value(value):
    """
    Convert a normalised Yelp value into a deterministic
    string representation for KG entity identifiers.
    """

    if value is None:
        return None

    if isinstance(value, (bool, np.bool_)):
        return (
            "true"
            if bool(value)
            else "false"
        )

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    if isinstance(value, (float, np.floating)):

        if np.isnan(value):
            return None

        if float(value).is_integer():
            return str(int(value))

        return str(float(value))

    cleaned = str(value).strip()

    if cleaned == "":
        return None

    return cleaned

In [108]:
category_triples = (
    business_category_links[
        [
            "business_id",
            "category"
        ]
    ]
    .copy()
)


category_triples[
    "head"
] = (
    "business::"
    +
    category_triples[
        "business_id"
    ].astype(str)
)


category_triples[
    "relation"
] = "has_category"


category_triples[
    "tail"
] = (
    "category::"
    +
    category_triples[
        "category"
    ].astype(str)
)


category_triples[
    "head_type"
] = "business"

category_triples[
    "tail_type"
] = "category"

category_triples[
    "source"
] = "category"


category_triples = (
    category_triples[
        [
            "head",
            "relation",
            "tail",
            "head_type",
            "tail_type",
            "source"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


print(
    "Category triples:",
    f"{len(category_triples):,}"
)

Category triples: 12,203


In [109]:
scalar_kg_records = []

for row in (
    scalar_attribute_df
    .itertuples(index=False)
):

    value = row.normalised_value

    if value is None:
        continue

    kg_value = canonical_kg_value(
        value
    )

    if kg_value is None:
        continue

    attribute_name = str(
        row.attribute
    ).strip()

    scalar_kg_records.append({
        "head":
            f"business::{row.business_id}",

        "relation":
            f"attribute::{attribute_name}",

        "tail":
            (
                f"value::{attribute_name}"
                f"::{kg_value}"
            ),

        "head_type":
            "business",

        "tail_type":
            "attribute_value",

        "source":
            "scalar_attribute"
    })


scalar_attribute_triples = pd.DataFrame(
    scalar_kg_records,
    columns=[
        "head",
        "relation",
        "tail",
        "head_type",
        "tail_type",
        "source"
    ]
)


scalar_attribute_triples = (
    scalar_attribute_triples
    .drop_duplicates(
        subset=[
            "head",
            "relation",
            "tail"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Scalar attribute triples:",
    f"{len(scalar_attribute_triples):,}"
)

print(
    "Scalar relations:",
    scalar_attribute_triples[
        "relation"
    ].nunique()
)

print(
    "Scalar value entities:",
    scalar_attribute_triples[
        "tail"
    ].nunique()
)

Scalar attribute triples: 33,504
Scalar relations: 32
Scalar value entities: 72


In [110]:
nested_kg_records = []


for row in (
    personalisation_businesses[
        [
            "business_id",
            "attributes_parsed"
        ]
    ]
    .itertuples(
        index=False
    )
):

    business_id = row.business_id

    for parent_attribute, raw_value in (
        row.attributes_parsed.items()
    ):

        nested_dict = (
            parse_nested_attribute(
                raw_value
            )
        )

        if nested_dict is None:
            continue

        for nested_key, nested_raw_value in (
            nested_dict.items()
        ):

            normalised_value = (
                normalise_yelp_scalar(
                    nested_raw_value
                )
            )

            if normalised_value is None:
                continue

            kg_value = canonical_kg_value(
                normalised_value
            )

            if kg_value is None:
                continue

            parent_attribute = str(
                parent_attribute
            ).strip()

            nested_key = str(
                nested_key
            ).strip()

            nested_kg_records.append({
                "head":
                    f"business::{business_id}",

                "relation":
                    (
                        "nested_attribute::"
                        f"{parent_attribute}"
                        f"::{nested_key}"
                    ),

                "tail":
                    (
                        "value::"
                        f"{parent_attribute}"
                        f"::{nested_key}"
                        f"::{kg_value}"
                    ),

                "head_type":
                    "business",

                "tail_type":
                    "nested_attribute_value",

                "source":
                    "nested_attribute"
            })


nested_attribute_triples = pd.DataFrame(
    nested_kg_records,
    columns=[
        "head",
        "relation",
        "tail",
        "head_type",
        "tail_type",
        "source"
    ]
)


nested_attribute_triples = (
    nested_attribute_triples
    .drop_duplicates(
        subset=[
            "head",
            "relation",
            "tail"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Nested attribute triples:",
    f"{len(nested_attribute_triples):,}"
)

print(
    "Nested relations:",
    nested_attribute_triples[
        "relation"
    ].nunique()
)

print(
    "Nested value entities:",
    nested_attribute_triples[
        "tail"
    ].nunique()
)

Nested attribute triples: 38,307
Nested relations: 41
Nested value entities: 73


In [111]:
metadata_kg_triples = pd.concat(
    [
        category_triples,
        scalar_attribute_triples,
        nested_attribute_triples
    ],
    ignore_index=True
)


metadata_kg_triples = (
    metadata_kg_triples
    .drop_duplicates(
        subset=[
            "head",
            "relation",
            "tail"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Total candidate metadata triples:",
    f"{len(metadata_kg_triples):,}"
)

print(
    "Unique relations:",
    metadata_kg_triples[
        "relation"
    ].nunique()
)

print(
    "Unique tail entities:",
    metadata_kg_triples[
        "tail"
    ].nunique()
)

print(
    "Businesses represented:",
    metadata_kg_triples[
        "head"
    ].nunique()
)

Total candidate metadata triples: 84,014
Unique relations: 74
Unique tail entities: 328
Businesses represented: 2516


In [112]:
metadata_source_summary = (
    metadata_kg_triples[
        "source"
    ]
    .value_counts()
    .rename_axis(
        "source"
    )
    .reset_index(
        name="triple_count"
    )
)


metadata_source_summary[
    "percentage"
] = (
    metadata_source_summary[
        "triple_count"
    ]
    /
    len(
        metadata_kg_triples
    )
    * 100
).round(2)


metadata_source_summary

,source,triple_count,percentage
0,nested_attribute,38307,45.60
1,scalar_attribute,33504,39.88
2,category,12203,14.52


In [113]:
relation_frequency = (
    metadata_kg_triples
    .groupby(
        "relation"
    )
    .agg(
        triple_count=(
            "tail",
            "size"
        ),

        businesses=(
            "head",
            "nunique"
        ),

        unique_tail_entities=(
            "tail",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "businesses",
        ascending=False
    )
)


print(
    "Relations:",
    len(
        relation_frequency
    )
)

print(
    "Relations used by <5 businesses:",
    (
        relation_frequency[
            "businesses"
        ] < 5
    ).sum()
)

relation_frequency

Relations: 74
Relations used by <5 businesses: 10


,relation,triple_count,businesses,unique_tail_entities
32,has_category,12203,2516,183
7,attribute::BusinessAcceptsCreditCards,2393,2393,2
25,attribute::RestaurantsPriceRange2,2247,2247,4
52,nested_attribute::BusinessParking::valet,2216,2216,2
50,nested_attribute::BusinessParking::lot,2169,2169,2
...,...,...,...,...
56,nested_attribute::DietaryRestrictions::halal,1,1,1
55,nested_attribute::DietaryRestrictions::gluten-...,1,1,1
54,nested_attribute::DietaryRestrictions::dairy-free,1,1,1
22,attribute::RestaurantsCounterService,1,1,1


In [114]:
tail_entity_frequency = (
    metadata_kg_triples
    .groupby(
        [
            "tail",
            "tail_type"
        ]
    )
    .agg(
        business_count=(
            "head",
            "nunique"
        ),

        triple_count=(
            "head",
            "size"
        )
    )
    .reset_index()
    .sort_values(
        "business_count",
        ascending=False
    )
)


tail_frequency_audit = pd.Series({
    "Unique tail entities":
        tail_entity_frequency[
            "tail"
        ].nunique(),

    "Tail entities appearing once":
        (
            tail_entity_frequency[
                "business_count"
            ] == 1
        ).sum(),

    "Tail entities appearing in <5 businesses":
        (
            tail_entity_frequency[
                "business_count"
            ] < 5
        ).sum(),

    "Tail entities appearing in >=5 businesses":
        (
            tail_entity_frequency[
                "business_count"
            ] >= 5
        ).sum()
})


tail_frequency_audit

Unique tail entities                         328
Tail entities appearing once                  11
Tail entities appearing in <5 businesses      16
Tail entities appearing in >=5 businesses    312
dtype: int64

In [115]:
expected_business_heads = set(
    "business::"
    +
    personalisation_businesses[
        "business_id"
    ].astype(str)
)


actual_business_heads = set(
    metadata_kg_triples[
        "head"
    ]
)


metadata_kg_check = pd.Series({
    "Total metadata triples":
        len(
            metadata_kg_triples
        ),

    "Unique business heads":
        metadata_kg_triples[
            "head"
        ].nunique(),

    "Expected businesses":
        len(
            expected_business_heads
        ),

    "All businesses represented":
        (
            expected_business_heads
            ==
            actual_business_heads
        ),

    "Unique relations":
        metadata_kg_triples[
            "relation"
        ].nunique(),

    "Unique tail entities":
        metadata_kg_triples[
            "tail"
        ].nunique(),

    "Duplicate triples":
        metadata_kg_triples
        .duplicated(
            subset=[
                "head",
                "relation",
                "tail"
            ]
        )
        .sum(),

    "Missing heads":
        metadata_kg_triples[
            "head"
        ].isna().sum(),

    "Missing relations":
        metadata_kg_triples[
            "relation"
        ].isna().sum(),

    "Missing tails":
        metadata_kg_triples[
            "tail"
        ].isna().sum()
})


metadata_kg_check

Total metadata triples        84014
Unique business heads          2516
Expected businesses            2516
All businesses represented     True
Unique relations                 74
Unique tail entities            328
Duplicate triples                 0
Missing heads                     0
Missing relations                 0
Missing tails                     0
dtype: object

### Stage F2.2 — Final Metadata KG Prune

In [116]:
MIN_METADATA_ENTITY_FREQUENCY = 5


# --------------------------------------------------
# Identify valid metadata tail entities
# --------------------------------------------------

valid_tail_entities = set(
    tail_entity_frequency.loc[
        tail_entity_frequency[
            "business_count"
        ] >= MIN_METADATA_ENTITY_FREQUENCY,
        "tail"
    ]
)


# --------------------------------------------------
# Freeze metadata KG
# --------------------------------------------------

frozen_metadata_kg = (
    metadata_kg_triples[
        metadata_kg_triples[
            "tail"
        ].isin(
            valid_tail_entities
        )
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "Original candidate triples:",
    f"{len(metadata_kg_triples):,}"
)

print(
    "Frozen metadata triples:",
    f"{len(frozen_metadata_kg):,}"
)

print(
    "Triples removed:",
    f"{len(metadata_kg_triples) - len(frozen_metadata_kg):,}"
)

print(
    "Final unique relations:",
    frozen_metadata_kg[
        "relation"
    ].nunique()
)

print(
    "Final tail entities:",
    frozen_metadata_kg[
        "tail"
    ].nunique()
)

print(
    "Business coverage maintained:",
    frozen_metadata_kg[
        "head"
    ].nunique()
    == 2516
)

Original candidate triples: 84,014
Frozen metadata triples: 83,989
Triples removed: 25
Final unique relations: 63
Final tail entities: 312
Business coverage maintained: True


In [117]:
frozen_metadata_kg_check = pd.Series({
    "Metadata triples":
        len(
            frozen_metadata_kg
        ),

    "Business nodes":
        frozen_metadata_kg[
            "head"
        ].nunique(),

    "Expected businesses":
        2516,

    "Relation types":
        frozen_metadata_kg[
            "relation"
        ].nunique(),

    "Tail entities":
        frozen_metadata_kg[
            "tail"
        ].nunique(),

    "Duplicate triples":
        frozen_metadata_kg
        .duplicated(
            subset=[
                "head",
                "relation",
                "tail"
            ]
        )
        .sum(),

    "Missing heads":
        frozen_metadata_kg[
            "head"
        ].isna().sum(),

    "Missing relations":
        frozen_metadata_kg[
            "relation"
        ].isna().sum(),

    "Missing tails":
        frozen_metadata_kg[
            "tail"
        ].isna().sum(),

    "All businesses represented":
        (
            frozen_metadata_kg[
                "head"
            ].nunique()
            == 2516
        )
})

frozen_metadata_kg_check

Metadata triples              83989
Business nodes                 2516
Expected businesses            2516
Relation types                   63
Tail entities                   312
Duplicate triples                 0
Missing heads                     0
Missing relations                 0
Missing tails                     0
All businesses represented     True
dtype: object

In [118]:
metadata_kg_dir = (
    processed_data_dir.parent
    / "new_orleans_knowledge_graph"
)

metadata_kg_dir.mkdir(
    parents=True,
    exist_ok=True
)


frozen_metadata_kg_path = (
    metadata_kg_dir
    / "new_orleans_frozen_metadata_kg.parquet"
)


frozen_metadata_kg.to_parquet(
    frozen_metadata_kg_path,
    index=False,
    engine="pyarrow"
)


print(
    "Frozen metadata KG saved:",
    frozen_metadata_kg_path.exists()
)

print(
    "Path:",
    frozen_metadata_kg_path
)

Frozen metadata KG saved: True
Path: processed_data/new_orleans_knowledge_graph/new_orleans_frozen_metadata_kg.parquet


## Stage F2.3 — Training-Only User–Business Interaction Graph

User–business interaction edges are constructed exclusively from the training
partition of the positive implicit-feedback dataset.

Each training interaction is represented as an `interacted_with` relation
between a user node and a business node. Validation and test interactions are
excluded from graph construction and retained solely for recommendation
evaluation.

This separation prevents held-out recommendation targets from leaking into the
knowledge graph while preserving the full training interaction structure.

In [119]:
# --------------------------------------------------
# Locate frozen interaction split files
# --------------------------------------------------

search_root = processed_data_dir.parent

candidate_interaction_files = []

for path in sorted(
    search_root.rglob("*.parquet")
):
    path_text = str(path).lower()

    # Ignore archived/intermediate material
    if "archive" in path_text:
        continue

    filename = path.name.lower()

    if (
        "interaction" in filename
        and any(
            split_name in filename
            for split_name in [
                "train",
                "validation",
                "test"
            ]
        )
    ):
        candidate_interaction_files.append(
            path
        )


print(
    "Candidate interaction split files:"
)

for path in candidate_interaction_files:
    print(path)

Candidate interaction split files:
processed_data/new_orleans_knowledge_graph/new_orleans_training_interaction_kg.parquet


In [120]:
# --------------------------------------------------
# Locate all likely frozen train/validation/test files
# --------------------------------------------------

search_root = processed_data_dir.parent

candidate_split_files = []

for path in sorted(search_root.rglob("*")):

    if not path.is_file():
        continue

    path_text = str(path).lower()

    # Ignore archived/intermediate copies
    if "archive" in path_text:
        continue

    filename = path.name.lower()

    # Search all common tabular formats
    if path.suffix.lower() not in {
        ".parquet",
        ".csv",
        ".pkl",
        ".pickle"
    }:
        continue

    if any(
        token in filename
        for token in [
            "train",
            "validation",
            "valid",
            "val",
            "test"
        ]
    ):
        candidate_split_files.append(path)


print(
    "Candidate split files:",
    len(candidate_split_files)
)

for path in candidate_split_files:
    print(path)

Candidate split files: 9
processed_data/new_orleans_image_pipeline/new_orleans_validated_photo_inventory.parquet
processed_data/new_orleans_knowledge_graph/new_orleans_training_interaction_kg.parquet
processed_data/new_orleans_subset/new_orleans_positive_test.parquet
processed_data/new_orleans_subset/new_orleans_positive_train.parquet
processed_data/new_orleans_subset/new_orleans_positive_validation.parquet
processed_data/new_orleans_text_pipeline/bge_small_en_v1_5_embeddings/new_orleans_bge_training_business_embedding_index.parquet
processed_data/new_orleans_text_pipeline/bge_small_en_v1_5_embeddings/new_orleans_bge_training_chunk_embedding_index.parquet
processed_data/new_orleans_text_pipeline/bge_small_en_v1_5_embeddings/new_orleans_bge_training_review_embedding_index.parquet
processed_data/new_orleans_text_pipeline/new_orleans_training_review_chunk_manifest.parquet


In [121]:
# --------------------------------------------------
# Load frozen positive interaction splits
# --------------------------------------------------

train_interactions = pd.read_parquet(
    processed_data_dir
    / "new_orleans_positive_train.parquet"
)

validation_interactions = pd.read_parquet(
    processed_data_dir
    / "new_orleans_positive_validation.parquet"
)

test_interactions = pd.read_parquet(
    processed_data_dir
    / "new_orleans_positive_test.parquet"
)


print(
    "Train interactions:",
    f"{len(train_interactions):,}"
)

print(
    "Validation interactions:",
    f"{len(validation_interactions):,}"
)

print(
    "Test interactions:",
    f"{len(test_interactions):,}"
)

Train interactions: 122,233
Validation interactions: 14,991
Test interactions: 14,991


In [122]:
assert len(train_interactions) == 122233
assert len(validation_interactions) == 14991
assert len(test_interactions) == 14991

required_columns = {
    "user_id",
    "business_id"
}

for split_name, split_df in {
    "train": train_interactions,
    "validation": validation_interactions,
    "test": test_interactions
}.items():

    missing_columns = (
        required_columns
        -
        set(split_df.columns)
    )

    assert not missing_columns, (
        f"{split_name} is missing columns: "
        f"{missing_columns}"
    )

print(
    "Frozen interaction splits loaded and verified."
)

Frozen interaction splits loaded and verified.


In [123]:
interaction_split_summary = pd.DataFrame({
    "split": [
        "train",
        "validation",
        "test"
    ],

    "interactions": [
        len(train_interactions),
        len(validation_interactions),
        len(test_interactions)
    ],

    "users": [
        train_interactions[
            "user_id"
        ].nunique(),

        validation_interactions[
            "user_id"
        ].nunique(),

        test_interactions[
            "user_id"
        ].nunique()
    ],

    "businesses": [
        train_interactions[
            "business_id"
        ].nunique(),

        validation_interactions[
            "business_id"
        ].nunique(),

        test_interactions[
            "business_id"
        ].nunique()
    ]
})

interaction_split_summary

,split,interactions,users,businesses
0,train,122233,14991,2516
1,validation,14991,14991,1966
2,test,14991,14991,1966


In [124]:
train_pair_duplicates = (
    train_interactions
    .duplicated(
        subset=[
            "user_id",
            "business_id"
        ]
    )
    .sum()
)

unique_train_pairs = (
    train_interactions[
        [
            "user_id",
            "business_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Training interactions:",
    f"{len(train_interactions):,}"
)

print(
    "Unique user-business pairs:",
    f"{unique_train_pairs:,}"
)

print(
    "Duplicate user-business pairs:",
    train_pair_duplicates
)

Training interactions: 122,233
Unique user-business pairs: 122,233
Duplicate user-business pairs: 0


In [125]:
expected_business_ids = set(
    personalisation_businesses[
        "business_id"
    ]
)

train_business_ids = set(
    train_interactions[
        "business_id"
    ]
)

business_universe_check = pd.Series({
    "Expected personalisation businesses":
        len(expected_business_ids),

    "Training businesses":
        len(train_business_ids),

    "Missing from training":
        len(
            expected_business_ids
            -
            train_business_ids
        ),

    "Unexpected training businesses":
        len(
            train_business_ids
            -
            expected_business_ids
        ),

    "Business sets exactly match":
        (
            expected_business_ids
            ==
            train_business_ids
        )
})

business_universe_check

Expected personalisation businesses    2516
Training businesses                    2516
Missing from training                     0
Unexpected training businesses            0
Business sets exactly match            True
dtype: object

In [126]:
train_pairs = set(
    map(
        tuple,
        train_interactions[
            [
                "user_id",
                "business_id"
            ]
        ].to_numpy()
    )
)

validation_pairs = set(
    map(
        tuple,
        validation_interactions[
            [
                "user_id",
                "business_id"
            ]
        ].to_numpy()
    )
)

test_pairs = set(
    map(
        tuple,
        test_interactions[
            [
                "user_id",
                "business_id"
            ]
        ].to_numpy()
    )
)

interaction_leakage_check = pd.Series({
    "Train pairs":
        len(train_pairs),

    "Validation pairs":
        len(validation_pairs),

    "Test pairs":
        len(test_pairs),

    "Train-validation overlap":
        len(
            train_pairs
            &
            validation_pairs
        ),

    "Train-test overlap":
        len(
            train_pairs
            &
            test_pairs
        ),

    "Validation-test overlap":
        len(
            validation_pairs
            &
            test_pairs
        )
})

interaction_leakage_check

Train pairs                 122233
Validation pairs             14991
Test pairs                   14991
Train-validation overlap         0
Train-test overlap               0
Validation-test overlap          0
dtype: int64

### Stage F2.3 — Construct and freeze training interaction triples

In [127]:
training_interaction_kg = (
    train_interactions[
        [
            "user_id",
            "business_id"
        ]
    ]
    .copy()
)


training_interaction_kg[
    "head"
] = (
    "user::"
    +
    training_interaction_kg[
        "user_id"
    ].astype(str)
)


training_interaction_kg[
    "relation"
] = "interacted_with"


training_interaction_kg[
    "tail"
] = (
    "business::"
    +
    training_interaction_kg[
        "business_id"
    ].astype(str)
)


training_interaction_kg[
    "head_type"
] = "user"

training_interaction_kg[
    "tail_type"
] = "business"

training_interaction_kg[
    "source"
] = "training_interaction"


training_interaction_kg = (
    training_interaction_kg[
        [
            "head",
            "relation",
            "tail",
            "head_type",
            "tail_type",
            "source"
        ]
    ]
    .drop_duplicates(
        subset=[
            "head",
            "relation",
            "tail"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Training interaction triples:",
    f"{len(training_interaction_kg):,}"
)

print(
    "User nodes:",
    f"{training_interaction_kg['head'].nunique():,}"
)

print(
    "Business nodes:",
    f"{training_interaction_kg['tail'].nunique():,}"
)

Training interaction triples: 122,233
User nodes: 14,991
Business nodes: 2,516


In [128]:
metadata_business_nodes = set(
    frozen_metadata_kg[
        "head"
    ]
)

interaction_business_nodes = set(
    training_interaction_kg[
        "tail"
    ]
)


business_node_alignment_check = pd.Series({
    "Metadata business nodes":
        len(
            metadata_business_nodes
        ),

    "Interaction business nodes":
        len(
            interaction_business_nodes
        ),

    "Missing from interaction graph":
        len(
            metadata_business_nodes
            -
            interaction_business_nodes
        ),

    "Missing from metadata graph":
        len(
            interaction_business_nodes
            -
            metadata_business_nodes
        ),

    "Business node sets exactly match":
        (
            metadata_business_nodes
            ==
            interaction_business_nodes
        )
})

business_node_alignment_check

Metadata business nodes             2516
Interaction business nodes          2516
Missing from interaction graph         0
Missing from metadata graph            0
Business node sets exactly match    True
dtype: object

In [129]:
user_training_degree = (
    training_interaction_kg
    .groupby(
        "head"
    )
    .size()
)


business_training_degree = (
    training_interaction_kg
    .groupby(
        "tail"
    )
    .size()
)


training_degree_summary = pd.Series({
    "Users":
        len(
            user_training_degree
        ),

    "Minimum user train interactions":
        user_training_degree.min(),

    "Median user train interactions":
        user_training_degree.median(),

    "Mean user train interactions":
        user_training_degree.mean(),

    "Maximum user train interactions":
        user_training_degree.max(),

    "Businesses":
        len(
            business_training_degree
        ),

    "Minimum business train interactions":
        business_training_degree.min(),

    "Median business train interactions":
        business_training_degree.median(),

    "Mean business train interactions":
        business_training_degree.mean(),

    "Maximum business train interactions":
        business_training_degree.max()
})

training_degree_summary

Users                                  14991.000000
Minimum user train interactions            3.000000
Median user train interactions             5.000000
Mean user train interactions               8.153759
Maximum user train interactions          661.000000
Businesses                              2516.000000
Minimum business train interactions        1.000000
Median business train interactions        19.000000
Mean business train interactions          48.582273
Maximum business train interactions     1271.000000
dtype: float64

In [130]:
training_interaction_kg_check = pd.Series({
    "Interaction triples":
        len(
            training_interaction_kg
        ),

    "Expected train interactions":
        len(
            train_interactions
        ),

    "All train interactions preserved":
        (
            len(
                training_interaction_kg
            )
            ==
            len(
                train_interactions
            )
        ),

    "User nodes":
        training_interaction_kg[
            "head"
        ].nunique(),

    "Expected users":
        14991,

    "Business nodes":
        training_interaction_kg[
            "tail"
        ].nunique(),

    "Expected businesses":
        2516,

    "Unique relations":
        training_interaction_kg[
            "relation"
        ].nunique(),

    "Duplicate triples":
        training_interaction_kg
        .duplicated(
            subset=[
                "head",
                "relation",
                "tail"
            ]
        )
        .sum(),

    "Missing heads":
        training_interaction_kg[
            "head"
        ].isna().sum(),

    "Missing relations":
        training_interaction_kg[
            "relation"
        ].isna().sum(),

    "Missing tails":
        training_interaction_kg[
            "tail"
        ].isna().sum(),

    "Train-validation leakage":
        len(
            train_pairs
            &
            validation_pairs
        ),

    "Train-test leakage":
        len(
            train_pairs
            &
            test_pairs
        ),

    "Validation-test overlap":
        len(
            validation_pairs
            &
            test_pairs
        ),

    "Business sets align with metadata KG":
        (
            metadata_business_nodes
            ==
            interaction_business_nodes
        )
})

training_interaction_kg_check

Interaction triples                     122233
Expected train interactions             122233
All train interactions preserved          True
User nodes                               14991
Expected users                           14991
Business nodes                            2516
Expected businesses                       2516
Unique relations                             1
Duplicate triples                            0
Missing heads                                0
Missing relations                            0
Missing tails                                0
Train-validation leakage                     0
Train-test leakage                           0
Validation-test overlap                      0
Business sets align with metadata KG      True
dtype: object

In [131]:
training_interaction_kg_path = (
    metadata_kg_dir
    / "new_orleans_training_interaction_kg.parquet"
)


training_interaction_kg.to_parquet(
    training_interaction_kg_path,
    index=False,
    engine="pyarrow"
)


print(
    "Training interaction KG saved:",
    training_interaction_kg_path.exists()
)

print(
    "Path:",
    training_interaction_kg_path
)

Training interaction KG saved: True
Path: processed_data/new_orleans_knowledge_graph/new_orleans_training_interaction_kg.parquet


## Stage F2.4 — Unified Model-Ready Knowledge Graph

The frozen structured metadata graph and training-only user–business interaction
graph are combined into a unified heterogeneous knowledge graph.

Deterministic integer identifiers are assigned to all entities and relations.
Business entities are ordered according to the previously frozen multimodal
business-feature index so that graph nodes, textual representations, visual
representations and recommendation candidates refer to exactly the same
business ordering.

The frozen graph contains only training interaction evidence. Validation and
test interactions remain excluded and are used exclusively during model
selection and final evaluation.

Inverse relations and self-loops, where required by the eventual graph model,
are not added to the frozen source graph. They can be generated explicitly at
model-training time while preserving the provenance of the original graph.

In [133]:
# --------------------------------------------------
# Locate frozen multimodal business feature indices
# --------------------------------------------------

search_root = processed_data_dir.parent


def find_unique_file(root, filename):
    matches = list(
        root.rglob(filename)
    )

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}"
        )

    if len(matches) > 1:
        print(
            f"Multiple matches found for {filename}:"
        )

        for match in matches:
            print(match)

        raise RuntimeError(
            "Multiple matching files found. "
            "Select the intended frozen file explicitly."
        )

    return matches[0]


visual_feature_index_path = find_unique_file(
    search_root,
    "new_orleans_personalisation_visual_feature_index.parquet"
)

text_feature_index_path = find_unique_file(
    search_root,
    "new_orleans_bge_training_business_embedding_index.parquet"
)


print(
    "Visual feature index:",
    visual_feature_index_path
)

print(
    "Text feature index:",
    text_feature_index_path
)

Visual feature index: processed_data/new_orleans_image_pipeline/clip_vit_b32_embeddings/new_orleans_personalisation_visual_feature_index.parquet
Text feature index: processed_data/new_orleans_text_pipeline/bge_small_en_v1_5_embeddings/new_orleans_bge_training_business_embedding_index.parquet


In [134]:
visual_feature_index = pd.read_parquet(
    visual_feature_index_path
)

text_feature_index = pd.read_parquet(
    text_feature_index_path
)


print(
    "\nVisual index rows:",
    len(visual_feature_index)
)

print(
    "Text index rows:",
    len(text_feature_index)
)

print(
    "\nVisual index columns:"
)

print(
    visual_feature_index.columns.tolist()
)

print(
    "\nText index columns:"
)

print(
    text_feature_index.columns.tolist()
)


Visual index rows: 2516
Text index rows: 2516

Visual index columns:
['business_id', 'visual_embedding_row', 'has_visual_feature', 'image_count', 'unique_image_labels', 'image_labels', 'aggregation_method']

Text index columns:
['business_id', 'visual_embedding_row', 'business_embedding_row', 'text_embedding_row', 'training_review_count', 'text_model', 'aggregation_method', 'training_reviewer_count', 'total_words', 'mean_review_words', 'median_review_words', 'maximum_review_words']


In [135]:
required_visual_columns = {
    "business_id",
    "visual_embedding_row"
}

required_text_columns = {
    "business_id",
    "text_embedding_row"
}


visual_missing = (
    required_visual_columns
    -
    set(
        visual_feature_index.columns
    )
)

text_missing = (
    required_text_columns
    -
    set(
        text_feature_index.columns
    )
)


print(
    "Missing required visual columns:",
    visual_missing
)

print(
    "Missing required text columns:",
    text_missing
)

Missing required visual columns: set()
Missing required text columns: set()


In [136]:
canonical_business_index = (
    visual_feature_index
    .sort_values(
        "visual_embedding_row"
    )
    .reset_index(drop=True)
    .copy()
)


canonical_business_ids = (
    canonical_business_index[
        "business_id"
    ]
    .astype(str)
    .to_numpy()
)


text_business_ids = (
    text_feature_index
    .sort_values(
        "text_embedding_row"
    )[
        "business_id"
    ]
    .astype(str)
    .to_numpy()
)


feature_alignment_check = pd.Series({
    "Visual businesses":
        len(
            canonical_business_ids
        ),

    "Text businesses":
        len(
            text_business_ids
        ),

    "Unique visual business IDs":
        len(
            set(
                canonical_business_ids
            )
        ),

    "Unique text business IDs":
        len(
            set(
                text_business_ids
            )
        ),

    "Text/visual business order identical":
        np.array_equal(
            canonical_business_ids,
            text_business_ids
        )
})

feature_alignment_check

Visual businesses                       2516
Text businesses                         2516
Unique visual business IDs              2516
Unique text business IDs                2516
Text/visual business order identical    True
dtype: object

In [137]:
canonical_user_ids = np.array(
    sorted(
        train_interactions[
            "user_id"
        ]
        .astype(str)
        .unique()
    )
)


user_nodes = np.array([
    f"user::{user_id}"
    for user_id
    in canonical_user_ids
])


print(
    "Canonical users:",
    len(
        canonical_user_ids
    )
)

Canonical users: 14991


In [138]:
business_nodes = np.array([
    f"business::{business_id}"
    for business_id
    in canonical_business_ids
])


print(
    "Canonical businesses:",
    len(
        business_nodes
    )
)

Canonical businesses: 2516


In [139]:
metadata_nodes = np.array(
    sorted(
        frozen_metadata_kg[
            "tail"
        ]
        .astype(str)
        .unique()
    )
)


print(
    "Metadata entities:",
    len(
        metadata_nodes
    )
)

Metadata entities: 312


In [140]:
entity_namespace_check = pd.Series({
    "User-business overlap":
        len(
            set(user_nodes)
            &
            set(business_nodes)
        ),

    "User-metadata overlap":
        len(
            set(user_nodes)
            &
            set(metadata_nodes)
        ),

    "Business-metadata overlap":
        len(
            set(business_nodes)
            &
            set(metadata_nodes)
        )
})

entity_namespace_check

User-business overlap        0
User-metadata overlap        0
Business-metadata overlap    0
dtype: int64

In [141]:
all_entities = np.concatenate([
    user_nodes,
    business_nodes,
    metadata_nodes
])


entity_to_id = {
    entity: entity_id
    for entity_id, entity
    in enumerate(
        all_entities
    )
}


print(
    "Total entities:",
    len(
        entity_to_id
    )
)

Total entities: 17819


In [142]:
metadata_entity_types = (
    frozen_metadata_kg[
        [
            "tail",
            "tail_type"
        ]
    ]
    .drop_duplicates()
)


metadata_type_count = (
    metadata_entity_types
    .groupby(
        "tail"
    )[
        "tail_type"
    ]
    .nunique()
)


assert (
    metadata_type_count.max()
    == 1
), (
    "At least one metadata entity "
    "has multiple tail types."
)


metadata_type_map = dict(
    zip(
        metadata_entity_types[
            "tail"
        ],
        metadata_entity_types[
            "tail_type"
        ]
    )
)

In [143]:
entity_index_records = []


for entity_id, entity in enumerate(
    all_entities
):

    if entity.startswith(
        "user::"
    ):

        entity_type = "user"

    elif entity.startswith(
        "business::"
    ):

        entity_type = "business"

    else:

        entity_type = (
            metadata_type_map[
                entity
            ]
        )

    entity_index_records.append({
        "entity_id":
            entity_id,

        "entity":
            entity,

        "entity_type":
            entity_type
    })


entity_index = pd.DataFrame(
    entity_index_records
)


entity_index[
    "entity_type"
].value_counts()

entity_type
user                      14991
business                   2516
category                    183
nested_attribute_value       65
attribute_value              64
Name: count, dtype: int64

In [144]:
metadata_relations = sorted(
    frozen_metadata_kg[
        "relation"
    ]
    .astype(str)
    .unique()
)


assert (
    "interacted_with"
    not in metadata_relations
)


all_relations = (
    ["interacted_with"]
    +
    metadata_relations
)


relation_to_id = {
    relation: relation_id
    for relation_id, relation
    in enumerate(
        all_relations
    )
}


relation_index = pd.DataFrame({
    "relation_id":
        np.arange(
            len(
                all_relations
            )
        ),

    "relation":
        all_relations
})


print(
    "Total relations:",
    len(
        relation_index
    )
)

relation_index.head()

Total relations: 64


,relation_id,relation
0,0,interacted_with
1,1,attribute::Alcohol
2,2,attribute::BYOB
3,3,attribute::BYOBCorkage
4,4,attribute::BikeParking


In [145]:
unified_kg = pd.concat(
    [
        training_interaction_kg,
        frozen_metadata_kg
    ],
    ignore_index=True
)


print(
    "Unified triples:",
    f"{len(unified_kg):,}"
)

Unified triples: 206,222


In [146]:
unified_kg[
    "head_id"
] = (
    unified_kg[
        "head"
    ]
    .map(
        entity_to_id
    )
)


unified_kg[
    "relation_id"
] = (
    unified_kg[
        "relation"
    ]
    .map(
        relation_to_id
    )
)


unified_kg[
    "tail_id"
] = (
    unified_kg[
        "tail"
    ]
    .map(
        entity_to_id
    )
)

In [147]:
id_mapping_missing_check = pd.Series({
    "Missing head IDs":
        unified_kg[
            "head_id"
        ].isna().sum(),

    "Missing relation IDs":
        unified_kg[
            "relation_id"
        ].isna().sum(),

    "Missing tail IDs":
        unified_kg[
            "tail_id"
        ].isna().sum()
})

id_mapping_missing_check

Missing head IDs        0
Missing relation IDs    0
Missing tail IDs        0
dtype: int64

In [148]:
unified_kg[
    [
        "head_id",
        "relation_id",
        "tail_id"
    ]
] = (
    unified_kg[
        [
            "head_id",
            "relation_id",
            "tail_id"
        ]
    ]
    .astype(
        np.int64
    )
)

In [149]:
user_entity_index = pd.DataFrame({
    "user_id":
        canonical_user_ids,

    "user_row":
        np.arange(
            len(
                canonical_user_ids
            )
        )
})


user_entity_index[
    "entity"
] = (
    "user::"
    +
    user_entity_index[
        "user_id"
    ]
)


user_entity_index[
    "entity_id"
] = (
    user_entity_index[
        "entity"
    ]
    .map(
        entity_to_id
    )
)


assert (
    user_entity_index[
        "entity_id"
    ].notna().all()
)

In [150]:
business_entity_index = pd.DataFrame({
    "business_id":
        canonical_business_ids,

    "business_row":
        np.arange(
            len(
                canonical_business_ids
            )
        )
})


business_entity_index[
    "entity"
] = (
    "business::"
    +
    business_entity_index[
        "business_id"
    ]
)


business_entity_index[
    "entity_id"
] = (
    business_entity_index[
        "entity"
    ]
    .map(
        entity_to_id
    )
)

In [151]:
text_row_lookup = (
    text_feature_index[
        [
            "business_id",
            "text_embedding_row"
        ]
    ]
    .copy()
)

text_row_lookup[
    "business_id"
] = (
    text_row_lookup[
        "business_id"
    ]
    .astype(str)
)


business_entity_index = (
    business_entity_index
    .merge(
        text_row_lookup,
        on="business_id",
        how="left",
        validate="one_to_one"
    )
)

In [152]:
visual_columns = [
    "business_id",
    "visual_embedding_row"
]

if (
    "has_visual_feature"
    in visual_feature_index.columns
):
    visual_columns.append(
        "has_visual_feature"
    )


visual_row_lookup = (
    visual_feature_index[
        visual_columns
    ]
    .copy()
)

visual_row_lookup[
    "business_id"
] = (
    visual_row_lookup[
        "business_id"
    ]
    .astype(str)
)


business_entity_index = (
    business_entity_index
    .merge(
        visual_row_lookup,
        on="business_id",
        how="left",
        validate="one_to_one"
    )
)

In [153]:
business_multimodal_alignment_check = pd.Series({
    "Businesses":
        len(
            business_entity_index
        ),

    "Unique business IDs":
        business_entity_index[
            "business_id"
        ].nunique(),

    "Unique KG entity IDs":
        business_entity_index[
            "entity_id"
        ].nunique(),

    "Missing KG entity IDs":
        business_entity_index[
            "entity_id"
        ].isna().sum(),

    "Missing text rows":
        business_entity_index[
            "text_embedding_row"
        ].isna().sum(),

    "Missing visual rows":
        business_entity_index[
            "visual_embedding_row"
        ].isna().sum(),

    "Business row equals text row":
        np.array_equal(
            business_entity_index[
                "business_row"
            ].to_numpy(),
            business_entity_index[
                "text_embedding_row"
            ].to_numpy()
        ),

    "Business row equals visual row":
        np.array_equal(
            business_entity_index[
                "business_row"
            ].to_numpy(),
            business_entity_index[
                "visual_embedding_row"
            ].to_numpy()
        )
})

business_multimodal_alignment_check

Businesses                        2516
Unique business IDs               2516
Unique KG entity IDs              2516
Missing KG entity IDs                0
Missing text rows                    0
Missing visual rows                  0
Business row equals text row      True
Business row equals visual row    True
dtype: object

In [154]:
unified_kg_check = pd.Series({
    "Total triples":
        len(
            unified_kg
        ),

    "Expected triples":
        (
            len(
                frozen_metadata_kg
            )
            +
            len(
                training_interaction_kg
            )
        ),

    "Entity count":
        len(
            entity_index
        ),

    "Expected entities":
        17819,

    "User entities":
        (
            entity_index[
                "entity_type"
            ]
            == "user"
        ).sum(),

    "Business entities":
        (
            entity_index[
                "entity_type"
            ]
            == "business"
        ).sum(),

    "Metadata entities":
        (
            ~entity_index[
                "entity_type"
            ].isin(
                [
                    "user",
                    "business"
                ]
            )
        ).sum(),

    "Relation count":
        len(
            relation_index
        ),

    "Expected relations":
        64,

    "Duplicate triples":
        unified_kg
        .duplicated(
            subset=[
                "head",
                "relation",
                "tail"
            ]
        )
        .sum(),

    "Missing graph IDs":
        unified_kg[
            [
                "head_id",
                "relation_id",
                "tail_id"
            ]
        ]
        .isna()
        .any()
        .any(),

    "Minimum entity ID":
        min(
            unified_kg[
                "head_id"
            ].min(),
            unified_kg[
                "tail_id"
            ].min()
        ),

    "Maximum entity ID":
        max(
            unified_kg[
                "head_id"
            ].max(),
            unified_kg[
                "tail_id"
            ].max()
        ),

    "Business feature alignment":
        (
            business_multimodal_alignment_check[
                "Business row equals text row"
            ]
            and
            business_multimodal_alignment_check[
                "Business row equals visual row"
            ]
        )
})

unified_kg_check

Total triples                 206222
Expected triples              206222
Entity count                   17819
Expected entities              17819
User entities                  14991
Business entities               2516
Metadata entities                312
Relation count                    64
Expected relations                64
Duplicate triples                  0
Missing graph IDs              False
Minimum entity ID                  0
Maximum entity ID              17818
Business feature alignment      True
dtype: object

## Stage F2.5 — Model-Input Knowledge Graph Freeze

The validated heterogeneous graph, entity mappings, relation mappings and
business/user alignment indices are saved as the frozen graph input bundle for
the recommendation experiments.

The business index provides the common alignment between graph entities,
textual features, visual features and recommendation candidates.

No validation or test interaction is included in the frozen graph. Model-level
augmentations such as inverse relations or self-loops are deliberately excluded
from the source graph and will be generated explicitly during model training.

In [155]:
model_kg_triples = (
    unified_kg[
        [
            "head_id",
            "relation_id",
            "tail_id"
        ]
    ]
    .to_numpy(
        dtype=np.int64
    )
)


print(
    "Model KG triple matrix:",
    model_kg_triples.shape
)

print(
    "dtype:",
    model_kg_triples.dtype
)

Model KG triple matrix: (206222, 3)
dtype: int64


In [156]:
model_kg_numeric_check = pd.Series({
    "Triple rows":
        model_kg_triples.shape[0],

    "Expected triple rows":
        len(
            unified_kg
        ),

    "Columns":
        model_kg_triples.shape[1],

    "Expected columns":
        3,

    "Integer dtype":
        np.issubdtype(
            model_kg_triples.dtype,
            np.integer
        ),

    "Minimum head ID":
        model_kg_triples[
            :, 0
        ].min(),

    "Maximum head ID":
        model_kg_triples[
            :, 0
        ].max(),

    "Minimum relation ID":
        model_kg_triples[
            :, 1
        ].min(),

    "Maximum relation ID":
        model_kg_triples[
            :, 1
        ].max(),

    "Minimum tail ID":
        model_kg_triples[
            :, 2
        ].min(),

    "Maximum tail ID":
        model_kg_triples[
            :, 2
        ].max(),

    "All entity IDs valid":
        (
            (
                model_kg_triples[
                    :, [0, 2]
                ] >= 0
            ).all()
            and
            (
                model_kg_triples[
                    :, [0, 2]
                ]
                < len(
                    entity_index
                )
            ).all()
        ),

    "All relation IDs valid":
        (
            (
                model_kg_triples[
                    :, 1
                ] >= 0
            ).all()
            and
            (
                model_kg_triples[
                    :, 1
                ]
                < len(
                    relation_index
                )
            ).all()
        )
})

model_kg_numeric_check

Triple rows               206222
Expected triple rows      206222
Columns                        3
Expected columns               3
Integer dtype               True
Minimum head ID                0
Maximum head ID            17506
Minimum relation ID            0
Maximum relation ID           63
Minimum tail ID            14991
Maximum tail ID            17818
All entity IDs valid        True
All relation IDs valid      True
dtype: object

In [157]:
business_degree_lookup = (
    business_training_degree
    .rename(
        "training_interaction_count"
    )
    .reset_index()
    .rename(
        columns={
            "tail":
                "entity"
        }
    )
)


business_entity_index = (
    business_entity_index
    .merge(
        business_degree_lookup,
        on="entity",
        how="left",
        validate="one_to_one"
    )
)


assert (
    business_entity_index[
        "training_interaction_count"
    ].notna().all()
)

In [158]:
model_input_dir = (
    processed_data_dir.parent
    / "new_orleans_model_inputs"
)

model_input_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [159]:
unified_kg_path = (
    model_input_dir
    / "new_orleans_unified_training_kg.parquet"
)

unified_kg.to_parquet(
    unified_kg_path,
    index=False,
    engine="pyarrow"
)

In [160]:
entity_index_path = (
    model_input_dir
    / "new_orleans_kg_entity_index.parquet"
)

relation_index_path = (
    model_input_dir
    / "new_orleans_kg_relation_index.parquet"
)

user_entity_index_path = (
    model_input_dir
    / "new_orleans_user_entity_index.parquet"
)

business_entity_index_path = (
    model_input_dir
    / "new_orleans_business_entity_index.parquet"
)


entity_index.to_parquet(
    entity_index_path,
    index=False,
    engine="pyarrow"
)

relation_index.to_parquet(
    relation_index_path,
    index=False,
    engine="pyarrow"
)

user_entity_index.to_parquet(
    user_entity_index_path,
    index=False,
    engine="pyarrow"
)

business_entity_index.to_parquet(
    business_entity_index_path,
    index=False,
    engine="pyarrow"
)

In [162]:
# --------------------------------------------------
# Re-establish frozen model-input output paths
# --------------------------------------------------

model_input_dir = (
    processed_data_dir.parent
    / "new_orleans_model_inputs"
)

model_input_dir.mkdir(
    parents=True,
    exist_ok=True
)


unified_kg_path = (
    model_input_dir
    / "new_orleans_unified_training_kg.parquet"
)

model_kg_triples_path = (
    model_input_dir
    / "new_orleans_unified_training_kg_triples.npy"
)

entity_index_path = (
    model_input_dir
    / "new_orleans_kg_entity_index.parquet"
)

relation_index_path = (
    model_input_dir
    / "new_orleans_kg_relation_index.parquet"
)

user_entity_index_path = (
    model_input_dir
    / "new_orleans_user_entity_index.parquet"
)

business_entity_index_path = (
    model_input_dir
    / "new_orleans_business_entity_index.parquet"
)


print("Model input directory:")
print(model_input_dir)

print("\nOutput paths:")
print(unified_kg_path)
print(model_kg_triples_path)
print(entity_index_path)
print(relation_index_path)
print(user_entity_index_path)
print(business_entity_index_path)

Model input directory:
processed_data/new_orleans_model_inputs

Output paths:
processed_data/new_orleans_model_inputs/new_orleans_unified_training_kg.parquet
processed_data/new_orleans_model_inputs/new_orleans_unified_training_kg_triples.npy
processed_data/new_orleans_model_inputs/new_orleans_kg_entity_index.parquet
processed_data/new_orleans_model_inputs/new_orleans_kg_relation_index.parquet
processed_data/new_orleans_model_inputs/new_orleans_user_entity_index.parquet
processed_data/new_orleans_model_inputs/new_orleans_business_entity_index.parquet


In [163]:
model_bundle_save_check = pd.Series({
    "Unified KG":
        unified_kg_path.exists(),

    "Numerical KG triples":
        model_kg_triples_path.exists(),

    "Entity index":
        entity_index_path.exists(),

    "Relation index":
        relation_index_path.exists(),

    "User index":
        user_entity_index_path.exists(),

    "Business alignment index":
        business_entity_index_path.exists()
})

model_bundle_save_check

Unified KG                   True
Numerical KG triples        False
Entity index                 True
Relation index               True
User index                   True
Business alignment index     True
dtype: bool

In [165]:
# --------------------------------------------------
# Recover and save numerical KG triple matrix
# --------------------------------------------------

from pathlib import Path
import numpy as np


model_input_dir = (
    processed_data_dir.parent
    / "new_orleans_model_inputs"
)

model_input_dir.mkdir(
    parents=True,
    exist_ok=True
)


model_kg_triples_path = (
    model_input_dir
    / "new_orleans_unified_training_kg_triples.npy"
)


# Reconstruct directly from the validated unified KG
model_kg_triples = (
    unified_kg[
        [
            "head_id",
            "relation_id",
            "tail_id"
        ]
    ]
    .to_numpy(
        dtype=np.int64
    )
)


print(
    "Triple matrix shape:",
    model_kg_triples.shape
)

print(
    "Triple matrix dtype:",
    model_kg_triples.dtype
)


# Strong checks before saving
assert (
    model_kg_triples.shape
    ==
    (206222, 3)
)

assert np.issubdtype(
    model_kg_triples.dtype,
    np.integer
)


# Explicit save
np.save(
    str(model_kg_triples_path),
    model_kg_triples
)


print(
    "\nSaved:",
    model_kg_triples_path.exists()
)

print(
    "Path:",
    model_kg_triples_path
)

print(
    "File size:",
    f"{model_kg_triples_path.stat().st_size:,} bytes"
    if model_kg_triples_path.exists()
    else "FILE NOT FOUND"
)

Triple matrix shape: (206222, 3)
Triple matrix dtype: int64

Saved: True
Path: processed_data/new_orleans_model_inputs/new_orleans_unified_training_kg_triples.npy
File size: 4,949,456 bytes


In [166]:
reloaded_kg = pd.read_parquet(
    unified_kg_path
)

reloaded_triples = np.load(
    model_kg_triples_path
)

reloaded_entities = pd.read_parquet(
    entity_index_path
)

reloaded_relations = pd.read_parquet(
    relation_index_path
)

reloaded_users = pd.read_parquet(
    user_entity_index_path
)

reloaded_businesses = pd.read_parquet(
    business_entity_index_path
)

In [167]:
model_bundle_roundtrip_check = pd.Series({
    "KG rows preserved":
        len(
            reloaded_kg
        )
        == 206222,

    "Triple matrix shape preserved":
        (
            reloaded_triples.shape
            ==
            (206222, 3)
        ),

    "Triple matrix exactly preserved":
        np.array_equal(
            model_kg_triples,
            reloaded_triples
        ),

    "Entity count preserved":
        len(
            reloaded_entities
        )
        == 17819,

    "Relation count preserved":
        len(
            reloaded_relations
        )
        == 64,

    "User count preserved":
        len(
            reloaded_users
        )
        == 14991,

    "Business count preserved":
        len(
            reloaded_businesses
        )
        == 2516,

    "Business rows preserved":
        np.array_equal(
            reloaded_businesses[
                "business_row"
            ].to_numpy(),
            np.arange(2516)
        ),

    "Text alignment preserved":
        np.array_equal(
            reloaded_businesses[
                "text_embedding_row"
            ].to_numpy(),
            np.arange(2516)
        ),

    "Visual alignment preserved":
        np.array_equal(
            reloaded_businesses[
                "visual_embedding_row"
            ].to_numpy(),
            np.arange(2516)
        )
})

model_bundle_roundtrip_check

KG rows preserved                  True
Triple matrix shape preserved      True
Triple matrix exactly preserved    True
Entity count preserved             True
Relation count preserved           True
User count preserved               True
Business count preserved           True
Business rows preserved            True
Text alignment preserved           True
Visual alignment preserved         True
dtype: bool